# Cell Analysis 

## Concepts and work flow : 

Using the developed tools we will do the following for one cell : 
1. load data 
2. segment data and process number of blocks 
3. decide on KPIs to extract 
4. Extract KPIs for each cycle 
5. visualize KPIs for the cell


## Imports and Data preparation and processing 

In [1]:
import sys
import os
import yaml
import pandas as pd 


# Add src/ to sys.path so you can import batterydata
sys.path.append(os.path.abspath('../src'))

from batterydata.parsers.basytec_reader import BasyTecReader
from batterydata.pipeline.preprocessor import Preprocessor

In [3]:


with open('../local_config.yaml', 'r') as f:
    config = yaml.safe_load(f)
file_path = config['basytec_sample_file']


# Load data
reader = BasyTecReader(filepath=file_path, sample_id="demo")
result = reader.read()
df = result['data']




In [4]:
# Instantiate Preprocessor and process
preproc = Preprocessor(df, cell_capacity_ah=1.3)
seg, doe, blocks = preproc.process_with_blocks(min_block_len=4, max_block_len=4, min_repeats=2)
tables = preproc.export_tables()
segmented_df = tables["segmented_data"]
doe_table = tables["doe_table"]
metadata_table = tables["metadata"]

In [16]:
try:
    pd.testing.assert_frame_equal(segmented_df, seg)
    print("DataFrames are exactly equal")
except AssertionError as e:
    print("DataFrames differ:", e)



DataFrames are exactly equal


In [4]:
from typing import List, Dict

def pretty_print_blocks(blocks: List[Dict[str, object]]) -> None:
    """Print a readable summary of detected blocks.

    Parameters
    ----------
    blocks : list of dict
        Output from detect_repeating_blocks_with_steps() or find_repeating_blocks().
    """
    for b in blocks:
        step = b['start']
        if b['count'] > 1:
            print(f"Step {step}: {b['count']} x ({', '.join(b['block'])})")
        else:
            print(f"Step {step}: {', '.join(b['block'])}")


In [17]:
pretty_print_blocks(blocks)

Step 0: DIS_1.00_V3.0_noCV
Step 1: OCV_600
Step 2: 200 x (CH_1.00_V4.2_CV, OCV_10, DIS_1.00_V3.0_noCV, OCV_10)
Step 802: DIS_1.00_V3.0_CV
Step 803: OCV_10
Step 804: CH_0.05_V4.2_noCV
Step 805: OCV_10
Step 806: DIS_0.05_V3.0_CV
Step 807: 160 x (OCV_10, CH_1.00_V4.2_CV, OCV_10, DIS_1.00_V3.0_noCV)
Step 1447: OCV_10
Step 1448: CH_1.00_V3.8_CV
Step 1449: 35 x (CH_1.00_V4.2_CV, OCV_10, DIS_1.00_V3.0_noCV, OCV_10)
Step 1589: CH_1.00_V4.2_CV
Step 1590: OCV_10
Step 1591: DIS_1.00_V3.9_CV


This indicates that the cell had :
1. Discharge initial step 
2. Rest for 600 seconds
3. 200 x 
    1. charge CCCV
    2. rest 10 seconds
    3. Discharge CC 
    4. rest 10 seconds 
4. Discharge 
5. qOCV step : 
    1. Charge CCCV C/20
    2. rest 10 seconds 
    3. Discharge CC C/20


## Defining KPIs 



Lets define the following KPIs to be calculated for this test procedure. 

First of all for each cycle calculate 

<ol type='a'>
<li> charge capacity</li>
<li> discharge capcity </li>
<li> Energy charge </li>
<li> Energy discharge </li>
<li> Culombic Efficiency C/E % </li>
<li> Energy Efficiency E/E % </li>
<li> Capacity retention (Qi / Q1) </li>
<li> Total charge time </li>
<li> CV step duration</li>
<li> CC step duration </li>
<li> CV duration / total charge time </li>

</ol>


## KPI calculation using KPI engine 


In [5]:
# select a cycle : 
# from doe  teable, where block id = 2  
doe_cycle_df = doe[(doe['block_id']== 2 )]

# from segmented table select what was selected in deo table by step_number
raw_cycle_df = seg[(seg['step_number'].isin(doe_cycle_df['step_number'].tolist()))]

In [40]:
raw_cycle_df

,time_s,dataset_idx,step_time_s,set_time_s,line_idx,voltage_v,current_a,charge_ah,step_charge_ah,energy_wh,cycle_index,state_code,step_type,step_number,cc_cv
2325,2.318610e+03,2326,0.00451,2.318610e+03,6,3.169672,1.298755,-0.620380,0.000002,-2.282471,1,0,charge,4,CC
2326,2.319609e+03,2327,1.00335,2.319609e+03,6,3.188562,1.300026,-0.620019,0.000362,-2.281324,1,1,charge,4,CC
2327,2.320610e+03,2328,2.00413,2.320610e+03,6,3.199248,1.300008,-0.619658,0.000724,-2.280169,1,1,charge,4,CC
2328,2.321608e+03,2329,3.00282,2.321608e+03,6,3.208408,1.299999,-0.619297,0.001084,-2.279014,1,1,charge,4,CC
2329,2.322609e+03,2330,4.00382,2.322609e+03,6,3.216994,1.300017,-0.618936,0.001446,-2.277852,1,1,charge,4,CC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1446655,1.444588e+06,1446656,7.00345,1.444588e+06,9,3.031901,0.000000,-0.612063,0.000000,13.887679,200,1,OCV,1003,OCV
1446656,1.444589e+06,1446657,8.00360,1.444589e+06,9,3.033809,0.000000,-0.612063,0.000000,13.887679,200,1,OCV,1003,OCV
1446657,1.444590e+06,1446658,9.00361,1.444590e+06,9,3.035718,0.000000,-0.612063,0.000000,13.887679,200,1,OCV,1003,OCV
1446658,1.444591e+06,1446659,10.00000,1.444591e+06,9,3.037435,0.000000,-0.612063,0.000000,13.887679,200,2,OCV,1003,OCV


In [6]:
from batterydata.pipeline.kpi_calculators import *


In [7]:
# 1) choose KPIs
units = [
    QChargeAh(), QDischargeAh(),
    EChargeWh(), EDischargeWh(),
    CoulombicEfficiency(),           # from integrated raw
    CoulombicEfficiencyMeasured(),   # from device 'step_charge_ah' (if available)
]

# 2) table over many cycles
cycle_kpis = compute_multi_cycle_kpi_table(raw_cycle_df, doe_cycle_df, units)

# 3) add retention
cycle_kpis = add_capacity_retention(cycle_kpis)  # uses Q_discharge_Ah by default


In [8]:
cycle_kpis

,block_id,cycle_index,has_charge,has_discharge,is_partial_cycle,debug_stats,Q_charge_Ah,Q_discharge_Ah,E_charge_Wh,E_discharge_Wh,CoulombicEfficiency,CoulombicEfficiency_meas,cycle_number,CapacityRetention_%
0,2,1,True,True,False,"{'charge': {'runs': 1, 'rows': 3739, 'pairs': ...",1.250703,1.251205,4.873492,4.784121,1.000401,1.000407,1,100.000000
1,2,2,True,True,False,"{'charge': {'runs': 1, 'rows': 3764, 'pairs': ...",1.252091,1.251480,4.874264,4.785702,0.999512,0.999495,2,100.021942
2,2,3,True,True,False,"{'charge': {'runs': 1, 'rows': 3763, 'pairs': ...",1.251904,1.251410,4.873183,4.785740,0.999605,0.999617,3,100.016344
3,2,4,True,True,False,"{'charge': {'runs': 1, 'rows': 3766, 'pairs': ...",1.251879,1.251425,4.872892,4.785997,0.999638,0.999646,4,100.017605
4,2,5,True,True,False,"{'charge': {'runs': 1, 'rows': 3766, 'pairs': ...",1.251842,1.251384,4.872600,4.786030,0.999633,0.999650,5,100.014249
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,2,196,True,True,False,"{'charge': {'runs': 1, 'rows': 3718, 'pairs': ...",1.244288,1.243956,4.840493,4.759106,0.999733,0.999724,196,99.420598
196,2,197,True,True,False,"{'charge': {'runs': 1, 'rows': 3724, 'pairs': ...",1.244361,1.244002,4.840807,4.759353,0.999712,0.999722,197,99.424299
197,2,198,True,True,False,"{'charge': {'runs': 1, 'rows': 3728, 'pairs': ...",1.244413,1.244023,4.841055,4.759365,0.999686,0.999705,198,99.425992
198,2,199,True,True,False,"{'charge': {'runs': 1, 'rows': 3722, 'pairs': ...",1.244369,1.243996,4.840816,4.759194,0.999700,0.999703,199,99.423778


In [34]:
pd.Series(kpis)

block_id                                                               2
cycle_index                                                            1
has_charge                                                          True
has_discharge                                                       True
is_partial_cycle                                                   False
debug_stats            {'charge': {'runs': 1, 'rows': 3739, 'pairs': ...
Q_charge_Ah                                                     1.250703
Q_discharge_Ah                                                  1.251205
E_charge_Wh                                                     4.873492
E_discharge_Wh                                                  4.784121
CoulombicEfficiency                                             1.000401
dtype: object